In [52]:
import pandas as pd
#Sentiment Data
url1 = "https://drive.google.com/uc?id=1PgQC0tO8XN-wqkNyghWc_-mnrYv_nhSf"
sentiment = pd.read_csv(url1)

#Trades Data
url2 = "https://drive.google.com/uc?id=1IAfLZwu6rJzyWKgBToqwSmmVYU6VbjVs"
trades = pd.read_csv(url2)


In [88]:
sentiment['classification'].unique()

array(['Fear', 'Extreme Fear', 'Neutral', 'Greed', 'Extreme Greed'],
      dtype=object)

### PART A — DATA PREPARATION



In [87]:
print(sentiment.head())
print(trades.head())


    timestamp  value classification        date sentiment
0  1517463000     30           Fear  2018-02-01      Fear
1  1517549400     15   Extreme Fear  2018-02-02      Fear
2  1517635800     40           Fear  2018-02-03      Fear
3  1517722200     24   Extreme Fear  2018-02-04      Fear
4  1517808600     11   Extreme Fear  2018-02-05      Fear
                                      account  coin  execution_price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   

   size_tokens  size_usd side       timestamp_ist  start_position direction  \
0       986.87   7872.16  BUY 2024-12-02 22:50:00        0.000000       Buy   
1        16.00    127.68  BUY 2024-12-02 22:50:00      986.52

In [64]:
print("Sentiment shape:", sentiment.shape)
print("Trades shape:", trades.shape)

print("\nMissing:\n", trades.isnull().sum())
print("\nDuplicates:", trades.duplicated().sum())

Sentiment shape: (2644, 4)
Trades shape: (211224, 19)

Missing:
 account             0
coin                0
execution_price     0
size_tokens         0
size_usd            0
side                0
timestamp_ist       0
start_position      0
direction           0
closed_pnl          0
transaction_hash    0
order_id            0
crossed             0
fee                 0
trade_id            0
timestamp           0
date                0
classification      6
win                 0
dtype: int64

Duplicates: 0


Sentiment dataset contain rows=2644, columns=4 and 0 missing and duplicate value and also trades dataset has 211224 rows and 16 columns with 0 missing value and duplicates



In [55]:
#Change name of columns to lower case and replace spaces with underscores
trades.columns = trades.columns.str.strip().str.lower().str.replace(" ", "_")

In [66]:
sentiment['sentiment'] = sentiment['classification'].apply(
    lambda x: 'Fear' if 'Fear' in x else 'Greed'
)

In [67]:
trades['timestamp_ist'] = pd.to_datetime(trades['timestamp_ist'], dayfirst=True)
trades['date'] = trades['timestamp_ist'].dt.date

sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

In [68]:
trades = pd.merge(
    trades,
    sentiment[['date', 'sentiment']],
    on='date',
    how='left'
)

In [69]:
# Win
trades['win'] = trades['closed_pnl'] > 0


In [70]:
# Daily PnL
daily_pnl = trades.groupby(['date', 'account'])['closed_pnl'].sum()


In [71]:
# Trade size
trades.groupby('account')['size_usd'].mean()

account
0x083384f897ee0f19899168e3b1bec365f52a9012    16159.576734
0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd     1653.226327
0x271b280974205ca63b716753467d5a371de622ab     8893.000898
0x28736f43f1e871e6aa8b1148d38d4994275d72c4      507.626933
0x2c229d22b100a7beb69122eed721cee9b24011dd     3138.894782
0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891     1729.941104
0x39cef799f8b69da1995852eea189df24eb5cae3c     4790.575486
0x3f9a0aadc7f04a7c9d75dc1b5a6ddd6e36486cf6     3445.471265
0x420ab45e0bd8863569a5efbb9c05d91f40624641     5189.367128
0x430f09841d65beb3f27765503d0f850b8bce7713     2397.824753
0x47add9a56df66b524d5e2c1993a43cde53b6ed85      517.528924
0x4acb90e786d897ecffb614dc822eb231b4ffb9f4     9084.699093
0x4f93fead39b70a1824f981a54d4e55b278e9f760    17098.171055
0x513b8629fe877bb581bf244e326a047b249c4ff1    34396.580284
0x6d6a4b953f202f8df5bed40692e7fd865318264a      746.725651
0x72743ae2822edd658c0c50608fd7c5c501b2afbd     7216.667245
0x72c6a4624e1dffa724e6d00d64ceae698af892a0     2

In [80]:
# Trades per day
trades_per_day = trades.groupby(['date', 'sentiment']).size().reset_index(name='trades')


In [73]:
# Long/Short
trades['side'].value_counts(normalize=True)

side
SELL    0.513805
BUY     0.486195
Name: proportion, dtype: float64

#### Since leverage data was unavailable, trade size (USD) was used as a proxy for risk-taking behavior.

### PART B — ANALYSIS

#### 1. Performance: Fear vs Greed

In [74]:
trades.groupby('sentiment')['closed_pnl'].mean()


sentiment
Fear     49.212077
Greed    48.118246
Name: closed_pnl, dtype: float64

In [75]:
trades.groupby('sentiment')['win'].mean()


sentiment
Fear     0.407871
Greed    0.413444
Name: win, dtype: float64

In [76]:
trades.groupby('sentiment')['closed_pnl'].std()

sentiment
Fear     990.875398
Greed    867.308701
Name: closed_pnl, dtype: float64

Average PnL and win rates are nearly identical between Fear and Greed periods. However, PnL variability is significantly higher during Fear periods (~14%), indicating increased risk and less predictable outcomes.

#### 2. Behavioral Changes

In [77]:
trades.groupby('sentiment')['size_usd'].mean()

sentiment
Fear     7182.011019
Greed    4635.764077
Name: size_usd, dtype: float64

In [78]:
trades.groupby('sentiment')['side'].value_counts(normalize=True)

sentiment  side
Fear       SELL    0.504968
           BUY     0.495032
Greed      SELL    0.519577
           BUY     0.480423
Name: proportion, dtype: float64

In [81]:
trades_per_day.groupby('sentiment')['trades'].mean()

sentiment
Fear     792.733333
Greed    342.195187
Name: trades, dtype: float64

Traders exhibit higher risk-taking during Greed periods, characterized by larger position sizes and a stronger long bias. In contrast, Fear periods reflect more cautious and defensive trading behavior.

#### 3. Segmentation

##### Frequent vs Infrequent

In [82]:
trade_counts = trades.groupby('account').size()
threshold = trade_counts.median()

trades['trader_type'] = trades['account'].apply(
    lambda x: 'Frequent' if trade_counts[x] > threshold else 'Infrequent'
)

##### Size-Based Risk Groups

In [86]:
trades['size_group'] = pd.cut(
    trades['size_usd'],
    bins=[0,100,1000,5000,10000,50000],
    labels=['Very Small','Small','Medium','Large','Very Large']
)

##### Winners vs Losers

In [90]:
total_pnl = trades.groupby('account')['closed_pnl'].sum()

trades['trader_class'] = trades['account'].apply(
    lambda x: 'Winner' if total_pnl[x] > 0 else 'Loser'
)

🔥 📊 Insights (WRITE EXACTLY LIKE THIS)
💡 Insight 1:

Market sentiment does not significantly impact profitability but strongly affects risk, with Fear periods showing higher volatility.

💡 Insight 2:

Traders increase position sizes and exhibit stronger long bias during Greed periods, indicating overconfidence.

💡 Insight 3:

Frequent and lower-risk traders demonstrate more consistent performance, while high-risk traders show greater variability.

.

🔥 ✅ PART C — ACTIONABLE OUTPUT
💡 Strategy 1:

During Fear periods, reduce position sizes and avoid aggressive trades, as increased volatility leads to higher risk without improved returns.

💡 Strategy 2:

During Greed periods, maintain disciplined position sizing and avoid overexposure, as higher confidence does not significantly improve performance

In [91]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

trades['profit_flag'] = trades['closed_pnl'] > 0

X = trades[['size_usd']]
y = trades['profit_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.5953679506116729


In [92]:
from sklearn.cluster import KMeans

features = trades[['size_usd']]
kmeans = KMeans(n_clusters=3)
trades['cluster'] = kmeans.fit_predict(features)